In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import datasets
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

############################################
# CBAM MODULE
############################################

class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super(ChannelAttention, self).__init__()

        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.mlp = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // reduction, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_channels // reduction, in_channels, 1, bias=False)
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = self.mlp(self.avg_pool(x))
        max = self.mlp(self.max_pool(x))
        return self.sigmoid(avg + max)


class SpatialAttention(nn.Module):
    def __init__(self):
        super(SpatialAttention, self).__init__()

        self.conv = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        avg = torch.mean(x, dim=1, keepdim=True)
        max, _ = torch.max(x, dim=1, keepdim=True)

        x = torch.cat([avg, max], dim=1)

        return self.sigmoid(self.conv(x))


class CBAM(nn.Module):

    def __init__(self, channels):
        super(CBAM, self).__init__()

        self.channel = ChannelAttention(channels)
        self.spatial = SpatialAttention()

    def forward(self, x):

        x = x * self.channel(x)
        x = x * self.spatial(x)

        return x


############################################
# DERMANET MODEL (ConvNeXt + CBAM)
############################################

class DERMANet(nn.Module):

    def __init__(self, num_classes=3):
        super(DERMANet, self).__init__()

        convnext = torchvision.models.convnext_tiny(weights="DEFAULT")

        self.features = convnext.features

        self.cbam = CBAM(768)

        self.pool = nn.AdaptiveAvgPool2d((1,1))

        self.classifier = nn.Linear(768, num_classes)

    def forward(self, x):

        x = self.features(x)

        x = self.cbam(x)

        x = self.pool(x)

        x = torch.flatten(x, 1)

        x = self.classifier(x)

        return x


############################################
# DATA TRANSFORM
############################################

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
])

############################################
# LOAD TRAIN DATASET ONLY
############################################

dataset = datasets.ImageFolder(
    root= "/content/drive/MyDrive/NDC/train",    # change to your path
    transform=transform
)

targets = dataset.targets

############################################
# 5-FOLD CROSS VALIDATION
############################################

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

precision_scores = []
recall_scores = []
f1_scores = []

############################################
# CROSS VALIDATION LOOP
############################################

for fold, (train_ids, val_ids) in enumerate(kfold.split(np.arange(len(dataset)), targets)):

    print("\nFOLD", fold+1)

    train_sampler = torch.utils.data.SubsetRandomSampler(train_ids)
    val_sampler = torch.utils.data.SubsetRandomSampler(val_ids)

    train_loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=32,
        sampler=train_sampler
    )

    val_loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=32,
        sampler=val_sampler
    )

    model = DERMANet().to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=1e-4,
        weight_decay=1e-4
    )

    ############################################
    # TRAINING
    ############################################

    epochs = 50

    for epoch in range(epochs):

        model.train()

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

    ############################################
    # VALIDATION
    ############################################

    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)

            outputs = model(images)

            _, preds = torch.max(outputs, 1)

            y_true.extend(labels.numpy())
            y_pred.extend(preds.cpu().numpy())

    precision = precision_score(y_true, y_pred, average="macro")
    recall = recall_score(y_true, y_pred, average="macro")
    f1 = f1_score(y_true, y_pred, average="macro")

    precision_scores.append(precision)
    recall_scores.append(recall)
    f1_scores.append(f1)

############################################
# FINAL RESULTS
############################################

print("\nCross Validation Results")

print("Precision:", np.mean(precision_scores), "+/-", np.std(precision_scores))
print("Recall:", np.mean(recall_scores), "+/-", np.std(recall_scores))
print("F1 Score:", np.mean(f1_scores), "+/-", np.std(f1_scores))


FOLD 1
Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 188MB/s] 
